# HW4: Temperature Forecast Web App with CWA API

<span style="color:red">**Using the API key provided in this example for a one-time test is permitted.**</span> 

<span style="color:red">**However, the API key must not be used when submitting the assignment.**</span> 

<span style="color:red">**Doing so will result in a score of 0 for that part of the assignment.**</span>

## HW4-1: Fetch Weather Forecast Data
- Use the CWA API to fetch weather data for each region of Taiwan over a week.
- You must use the API that returns data in `JSON format`.
- The regions include:
    - 北部地區
    - 中部地區
    - 南部地區
    - 東北部地區
    - 東部地區
    - 東南部地區

### Step 1: Install the dependencies

In [1]:
%pip install -q requests streamlit pandas plotly


[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Step 2: Fetch weather forecast data

In [2]:
import json
import requests

# Set the URL for the API endpoint
url = "https://opendata.cwa.gov.tw/fileapi/v1/opendataapi/F-A0010-001?Authorization=CWA-3DE12743-8E82-4A80-9524-AC203C066CA5&downloadType=WEB&format=JSON"

# Fetch the data
response = requests.get(url)
data = response.json()

# Show the fetched data
print(json.dumps(data, indent=4, ensure_ascii=False))

{
    "cwaopendata": {
        "@xmlns": "urn:cwa:gov:tw:cwacommon:0.1",
        "identifier": "472c7047-eb48-4aa1-8fc0-9be46a234889",
        "datasetName": "一週農業氣象預報",
        "dataid": "F-A0010-001",
        "sender": "shchiang@cwa.gov.tw",
        "sent": "2025-04-12T18:00:03+08:00",
        "status": "Actual",
        "scope": "Public",
        "msgType": "Issue",
        "resources": {
            "resource": {
                "metadata": {
                    "resourceName": "中央氣象署氣候服務_農業氣象一週預報",
                    "resourceDescription": "提供氣象署預報資料和農事建議，供農業相關單位及農民參考",
                    "language": "zh",
                    "homepage": "https://www.cwa.gov.tw/V8/C/L/agriculture.html",
                    "sources": {
                        "source": [
                            {
                                "title": "1週預報",
                                "path": "https://www.cwa.gov.tw/V8/C/W/week.html"
                            },
                            {
      

## HW4-2: Analyze the Fetched Data to Locate Daily Max and Min Temperatures
- Analyze the data to locate the Daily `Max Temperatures` and `Min Temperatures`.
- You can do this manually or use ChatGPT for help.
- After analyzing, you will locate the information you need.

### Step 1: Analyze the data

In [3]:
pass

### Step 2: Retrieve only the Daily `MaxT` and `MinT` for Each Region
- Note that `Region` is referred to as `Location` in the dataset.

In [4]:
# Extract the temperature forecasts
temperature_forecasts = data["cwaopendata"]["resources"]["resource"]["data"]["agrWeatherForecasts"]["weatherForecasts"]["location"]

# Remove the "Wx" key from each region
for region in temperature_forecasts:
    if "Wx" in region["weatherElements"]:
        del region["weatherElements"]["Wx"]

# Print the data
print(json.dumps(temperature_forecasts, indent=4, ensure_ascii=False))

[
    {
        "locationName": "北部地區",
        "weatherElements": {
            "MaxT": {
                "daily": [
                    {
                        "dataDate": "2025-04-13",
                        "temperature": "21"
                    },
                    {
                        "dataDate": "2025-04-14",
                        "temperature": "28"
                    },
                    {
                        "dataDate": "2025-04-15",
                        "temperature": "26"
                    },
                    {
                        "dataDate": "2025-04-16",
                        "temperature": "27"
                    },
                    {
                        "dataDate": "2025-04-17",
                        "temperature": "28"
                    },
                    {
                        "dataDate": "2025-04-18",
                        "temperature": "29"
                    },
                    {
                        "d

## HW4-3: Save the Temperature Data to SQLite Database
- Create a SQLite database to store the temperature data.
- The database should include a table with the following columns:
    - `id`: Primary key
    - `regionName`: Region name
    - `dataDate`: Date of the forecast
    - `MaxT`: Maximum temperature
    - `MinT`: Minimum temperature
- After inserting the data, verify that it is stored correctly.

### Step 1: Save the data to SQlite database

In [5]:
import sqlite3

# Connect to SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect("data.db")
cursor = conn.cursor()

# Drop the old table if it exists
cursor.execute("DROP TABLE IF EXISTS TemperatureForecasts")

# Create a table to store the temperature forecasts
cursor.execute("""
CREATE TABLE TemperatureForecasts (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    regionName TEXT,
    dataDate INT,
    MaxT INT,
    MinT INT
)
""")

# Insert data into the table
for region in temperature_forecasts:
    region_name = region["locationName"]
    mint_data = region["weatherElements"]["MinT"]["daily"]
    maxt_data = region["weatherElements"]["MaxT"]["daily"]

    for mint, maxt in zip(mint_data, maxt_data):
        dataDate = mint["dataDate"]
        mint_value = int(mint["temperature"])
        maxt_value = int(maxt["temperature"])
        cursor.execute(
            "INSERT INTO TemperatureForecasts (regionName, dataDate, mint, maxt) VALUES (?, ?, ?, ?)",
            (region_name, dataDate, mint_value, maxt_value),
        )

# Commit changes and close the connection
conn.commit()
conn.close()

### Step 2: Check the data to ensure it is saved correctly

Query all the Region names.

In [6]:
import pandas as pd

# Reconnect to the SQLite database
conn = sqlite3.connect("data.db")

# Query to get all distinct region names
query = "SELECT DISTINCT regionName FROM TemperatureForecasts"
region_names = pd.read_sql_query(query, conn)

# Display the region names
print(region_names)

# Close the connection
conn.close()

  regionName
0       北部地區
1       中部地區
2       南部地區
3      東北部地區
4       東部地區
5      東南部地區


Query the temperature data for a specific Region, such as `中部地區`.

In [7]:
# Reconnect to the SQLite database
conn = sqlite3.connect("data.db")

# Load data into a pandas DataFrame
selected_region = "中部地區"
query = f"SELECT regionName, dataDate, MinT, MaxT FROM TemperatureForecasts WHERE regionName = '{selected_region}'"

df_weather_from_db = pd.read_sql_query(query, conn)

# Display the DataFrame
print(df_weather_from_db)

# Close the connection
conn.close()

  regionName    dataDate  MinT  MaxT
0       中部地區  2025-04-13    17    22
1       中部地區  2025-04-14    16    28
2       中部地區  2025-04-15    15    28
3       中部地區  2025-04-16    16    28
4       中部地區  2025-04-17    17    30
5       中部地區  2025-04-18    21    30
6       中部地區  2025-04-19    23    31


## HW4-4: Implement the Temperature Forecast Web App (Streamlit)
- Create a Streamlit app to visualize the temperature data.
- Allow users to select a region from a dropdown menu and view the temperature forecast for that region.
- The app `must` use SQL to query the data from the SQLite database.
- Include both a `line chart` and a `table` to display the temperature data over the week.


### Step 1: Create a streamlit app

In [8]:
%%writefile app.py

import streamlit as st
import sqlite3
import pandas as pd

import plotly.express as px

# Title of the app
st.title("Temperature Forecast Web App")

# Connect to the SQLite database
conn = sqlite3.connect("data.db")

# Query to get all distinct region names
query = "SELECT DISTINCT regionName FROM TemperatureForecasts"
region_names = pd.read_sql_query(query, conn)

# Dropdown to select region
selected_region = st.selectbox("Select a Region", region_names["regionName"])

# Query to get MaxT and MinT for the selected region
query = f"""
SELECT dataDate, MinT, MaxT
FROM TemperatureForecasts
WHERE regionName = '{selected_region}'
"""
df_weather = pd.read_sql_query(query, conn)

# Close the connection
conn.close()

# Plot the data using Plotly
fig = px.line(
    df_weather,
    x="dataDate",
    y=["MinT", "MaxT"],
    title=f"Temperature Trends for {selected_region}",
    labels={"value": "Temperature (°C)", "dataDate": "Date"},
)
st.plotly_chart(fig)

# Display the data
st.write(f"Temperature Data for {selected_region}")
st.dataframe(df_weather)

Overwriting app.py


### Step 2: Run the streamlit app

In [ ]:
# for colab
!echo "Tunnel Password: $(curl -s ifconfig.io)"
!streamlit run app.py &>logs.txt & \
  npx -y localtunnel --port 8501


# for local
# !streamlit run app.py